# Skeleton Lab: Multiple Linear Regression → Feature Selection → Shrinkage → Dimension Reduction

**Practice notebook — fill in the `# TODO` blanks.**

This mirrors the structure of `01_extended_lab_MLR_to_PCR.ipynb`, but the code is missing. Work through it
top-to-bottom without looking at the solutions notebook first. If you get stuck for more than a few minutes on a
step, peek at the corresponding section of `04_solutions_MLR_to_PCR.ipynb` — but try to write the line yourself
afterward rather than copy-pasting.

**Datasets used:**
- `sklearn.datasets.load_diabetes()` for Parts A & B (MLR + feature selection)
- A synthetic dataset from `sklearn.datasets.make_regression()` for Part C (shrinkage)
- `Hitters.csv` (in this folder) for Part D (PCR / PLS)

Each section states the **goal** and gives you the variable names your code should produce, so later cells work.


## 0. Setup
Import what you'll need. You will use: numpy, pandas, matplotlib, seaborn, statsmodels.api, sklearn (datasets, linear_model, model_selection, feature_selection, metrics, preprocessing, decomposition, cross_decomposition), and `variance_inflation_factor` from statsmodels.

In [ ]:
# TODO: import numpy, pandas, matplotlib.pyplot, seaborn
# TODO: import statsmodels.api as sm
# TODO: from statsmodels.stats.outliers_influence import variance_inflation_factor as vif
# TODO: from sklearn import datasets
# TODO: from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, ElasticNetCV, Ridge, Lasso, lasso_path
# TODO: from sklearn.model_selection import train_test_split, KFold, cross_val_score
# TODO: from sklearn.feature_selection import RFECV
# TODO: from sklearn.metrics import mean_squared_error, make_scorer
# TODO: from sklearn.preprocessing import StandardScaler
# TODO: from sklearn.decomposition import PCA
# TODO: from sklearn.cross_decomposition import PLSRegression

# TODO: set a random seed of 42 for reproducibility

# TODO: write a `rmse(y_true, y_pred)` helper function (sqrt of mean_squared_error)
# TODO: write a `mape(y_true, y_pred)` helper function: 100 * mean(|y_true - y_pred| / y_true)


## Part A — Multiple Linear Regression on the Diabetes Dataset

### 1. Load the data
Goal: create `X` (DataFrame of the 10 explanatory variables, with a `const` column of 1.0 prepended for the
statsmodels intercept) and `y` (Series/array of the target).

In [ ]:
# TODO: data = datasets.load_diabetes()
# TODO: y = data['target']
# TODO: X = pd.DataFrame(data['data'], columns=[...])   # use the 10 column names from the book:
#       'age','sex','bmi','bp','s1','s2','s3','s4','s5','s6'
# TODO: insert a 'const' column of 1.0 at position 0 in X


### 2. Scatter each feature against the target
Goal: a grid of scatter plots, one per feature (excluding `const`).

In [ ]:
# TODO: create a subplot grid and loop over the feature columns, scatter each vs. y


### 3. VIF-based multicollinearity reduction
Goal: write a function `reduce_by_vif(X_in, threshold=5.0, protect=('const',))` that:
1. Computes the VIF for every column.
2. Finds the highest VIF among columns *not* in `protect`.
3. If that max VIF ≥ `threshold`, drops that column and repeats.
4. Otherwise, stops and returns `(X_reduced, final_vif_series, removal_log)`.

Then call it to produce `X_reduced`.

In [ ]:
def reduce_by_vif(X_in, threshold=5.0, protect=('const',)):
    # TODO: implement the loop described above
    pass

# TODO: X_reduced, final_vifs, removal_log = reduce_by_vif(X, threshold=5.0)
# TODO: print what got removed, and print final_vifs


### 4. Fit OLS on `X_reduced` and print the summary

In [ ]:
# TODO: model = sm.OLS(y, X_reduced)
# TODO: fit = model.fit()
# TODO: print(fit.summary())


### 5. Residual diagnostics: histogram, residuals-vs-fitted, QQ-plot, Durbin-Watson

In [ ]:
# TODO: plot a histogram of fit.resid
# TODO: compute pred = fit.predict(X_reduced) and scatter pred vs. fit.resid with a horizontal line at 0
# TODO: sm.qqplot(fit.resid, line='45', fit=True) and plt.show()
# TODO: print the Durbin-Watson statistic: sm.stats.stattools.durbin_watson(fit.resid)


**Checkpoint A** — Write a short markdown answer (in a new markdown cell below) covering:
- Which assumptions look satisfied and which look borderline?
- Which coefficients are statistically significant (p < 0.05)? Interpret the largest one in plain English.

In [ ]:
# (no code needed here — add a markdown cell with your written answer)

## Part B — Feature Selection

### 1. Correlation ranking
Goal: a DataFrame `corr_with_target`, sorted by absolute correlation with the target, and a heatmap.

In [ ]:
# TODO: build a DataFrame `all_data` with the feature columns (no const) plus a 'target' column
# TODO: compute all_data.corr()[['target']], drop the 'target' row, sort by absolute value descending
# TODO: plot as a heatmap


### 2. Forward and backward selection from scratch
Goal: implement `forward_selection(X_in, y_in, threshold_in=0.05)` and
`backward_selection(X_in, y_in, threshold_out=0.05)` using p-values from `sm.OLS`, following the algorithm
description in the book (Chapter 7, 'Statistical significance').

In [ ]:
def forward_selection(X_in, y_in, threshold_in=0.05):
    # TODO: start with just 'const' selected (if present) and an empty 'remaining' list otherwise
    # TODO: at each step, try adding each remaining feature one at a time, fit OLS, record its p-value
    # TODO: add the feature with the lowest p-value IF that p-value is below threshold_in; else stop
    pass

def backward_selection(X_in, y_in, threshold_out=0.05):
    # TODO: start with all columns selected
    # TODO: at each step, fit OLS on the current selection, find the feature (excluding const) with the
    #       highest p-value; if it's above threshold_out, remove it and repeat; else stop
    pass

# TODO: run both and print the results


### 3. RFECV
Goal: run `RFECV` with a `LinearRegression` estimator, `cv=5`, and the MAPE as a (negated) scoring function.
Plot the cross-validated score against number of features.

In [ ]:
# TODO: X_no_const = X.drop(columns='const')
# TODO: fit a LinearRegression on X_no_const, y (not strictly required by RFECV but good practice)
# TODO: rfecv = RFECV(estimator=..., step=1, cv=5, scoring=make_scorer(mape, greater_is_better=False), min_features_to_select=1)
# TODO: rfecv.fit(X_no_const, y)
# TODO: print rfecv.n_features_ and the selected feature names
# TODO: plot -rfecv.cv_results_['mean_test_score'] vs. number of features


**Checkpoint B** — Compare the feature sets from VIF-reduction, forward selection, backward selection, and RFECV. Where do they agree/disagree, and why?

## Part C — Shrinkage Methods (Ridge / LASSO / Elastic Net)

### 1. Build a synthetic dataset
Goal: use `make_regression` to create `X_c` (DataFrame, 20 columns named `X1..X20`) and `y_c` (Series), with
`n_samples=2000`, `n_features=20`, `n_informative=8`, `effective_rank=6`, `noise=15.0`, `coef=True`,
`random_state=42`. Print how many features are truly informative (`true_coef != 0`).

In [ ]:
# TODO: from sklearn.datasets import make_regression
# TODO: X_c_raw, y_c, true_coef = make_regression(...)
# TODO: wrap into DataFrame/Series with column names X1..X20


### 2. Scale and split
Goal: `Xc_train, Xc_test, yc_train, yc_test` via `train_test_split(test_size=0.25, random_state=42)`, with
features scaled by a `StandardScaler` **fit on the training data only**.

In [ ]:
# TODO: scale X_c with StandardScaler (fit_transform), then split into train/test
# NOTE: for extra rigor, try re-doing this by splitting FIRST, then fitting the scaler on the training
#       split only and transforming both. Compare: does it change your results much here? Why/why not?


### 3. OLS baseline, Ridge (RidgeCV), LASSO (LassoCV), Elastic Net (ElasticNetCV)

In [ ]:
# TODO: fit LinearRegression on (Xc_train, yc_train); report train/test RMSE using your rmse() helper


In [ ]:
# TODO: alphas = np.logspace(-3, 3, 100)
# TODO: fit RidgeCV(alphas=alphas, cv=5); print the chosen alpha_; report train/test RMSE


In [ ]:
# TODO: fit LassoCV(alphas=alphas, cv=5, max_iter=20000); print chosen alpha_; report train/test RMSE
# TODO: also print how many coefficients were shrunk to exactly zero


In [ ]:
# TODO: fit ElasticNetCV(alphas=alphas, l1_ratio=[.1,.3,.5,.7,.9,.95,.99,1], cv=5, max_iter=20000)
# TODO: print chosen alpha_ and l1_ratio_; report train/test RMSE


### 4. Coefficient paths
Goal: for a range of alphas (e.g. `np.logspace(-3, 1, 60)`), fit a `Ridge` model at each alpha and collect
coefficients into an array; separately use `lasso_path` to get LASSO coefficients across alphas. Plot both as
line charts (one line per feature, x-axis = alpha on a log scale).

In [ ]:
# TODO: build ridge_coefs (list/array) by looping over alphas_path and fitting Ridge(alpha=a)
# TODO: use lasso_path(Xc_train, yc_train, alphas=alphas_path) to get lasso_coefs
# TODO: plot both coefficient paths side by side


**Checkpoint C** — Which features keep non-zero LASSO coefficients longest (largest alpha)? Do they match the truly informative features from `true_coef`?

## Part D — Dimension Reduction: PCR and PLS on the Hitters Dataset

### 1. Load & prepare
Goal: read `Hitters.csv`, drop rows with missing `Salary`, one-hot encode `League`, `Division`, `NewLeague`
(`drop_first=True`), concatenate with the numeric features to build `Xh`/`yh`, then split into
`Xh_train, Xh_test, yh_train, yh_test` (`test_size=0.2, random_state=42`).

> ⚠️ Fit your `StandardScaler` **only on `Xh_train`**, then `.transform()` both train and test. Do NOT fit a
> scaler or a PCA on the combined or test data — that is data leakage. (The original textbook example does this;
> your job here is to do it correctly.)

In [ ]:
# TODO: hitters = pd.read_csv('Hitters.csv').dropna()
# TODO: one-hot encode categorical columns with drop_first=True
# TODO: build Xh (numeric features + dummies) and yh (Salary)
# TODO: train_test_split
# TODO: fit StandardScaler on Xh_train ONLY; transform both train and test -> Xh_train_scaled, Xh_test_scaled


### 2. PCA scree plot on the training data

In [ ]:
# TODO: pca = PCA(); Xh_pc_train = pca.fit_transform(Xh_train_scaled)
# TODO: plot pca.explained_variance_ratio_ as a bar chart, and its cumulative sum as a line chart


### 3. Choose the number of components via 10-fold CV RMSE, using only the training PCA scores

In [ ]:
# TODO: loop over k = 1..num_components, cross_val_score a LinearRegression on Xh_pc_train[:, :k]
# TODO: pick best_k = argmin of the CV RMSE curve; plot the curve


### 4. Refit on best_k components; evaluate on test set
Remember: transform the test set with `pca.transform(...)` (not `fit_transform`!) using the *training* PCA object.

In [ ]:
# TODO: fit LinearRegression on Xh_pc_train[:, :best_k], yh_train
# TODO: Xh_pc_test = pca.transform(Xh_test_scaled)[:, :best_k]
# TODO: compute and print pcr_train_rmse, pcr_test_rmse


### 5. PLS Regression
Goal: for `n_components` from 1 to `min(15, num_features)`, cross-validate a `PLSRegression`, find the best
number of components, refit, and evaluate on the test set.

In [ ]:
# TODO: loop over n_components, cross_val_score PLSRegression, collect RMSEs
# TODO: pick best_k_pls, plot the curve
# TODO: refit PLSRegression(n_components=best_k_pls) on the training data
# TODO: compute pls_train_rmse, pls_test_rmse


## Part E — Final Comparison
Goal: build a small DataFrame `df_results` with rows OLS / Ridge / LASSO / ElasticNet / PCR / PLS, columns
`Train RMSE` and `Test RMSE`, all computed on the **same** Hitters train/test split (`Xh_train_scaled` /
`Xh_test_scaled`). Add a derived column for the overfitting gap, sort by test RMSE, and make a bar chart.

In [ ]:
# TODO: fit OLS, RidgeCV, LassoCV, ElasticNetCV on (Xh_train_scaled, yh_train)
# TODO: assemble results list of (name, train_rmse, test_rmse) tuples, including PCR and PLS from Part D
# TODO: build df_results, add 'Overfit gap' column, sort, display, and plot a grouped bar chart


**Checkpoint D (final)** — Answer:
1. Which model generalizes best?
2. Which has the largest overfitting gap?
3. Did PLS need fewer components than PCR? Why might that be?
4. Which model would you hand to a non-technical stakeholder for interpretability, and why?

When you're done, compare your notebook against `04_solutions_MLR_to_PCR.ipynb`.
